In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
try:
    # 尝试从 tf 内部调用 keras，如果还不行就直接 import keras
    from tensorflow import keras 
    print("✅ Keras 成功通过 TensorFlow 导入")
except ImportError:
    import keras
    print("✅ Keras 成功作为独立包导入")

print(f"TensorFlow 版本: {tf.__version__}")

✅ Keras 成功通过 TensorFlow 导入
TensorFlow 版本: 2.20.0


In [2]:
import numpy as np
import pandas as pd

In [3]:
# 从文件导入已有的函数
from preprocess import prepare_single_sample
from mediapipe_extractor import extract_landmarks_from_video

In [4]:
from model import ASLEnsemble

In [5]:
# 确保路径指向具体的子文件夹
MODEL_DIR = 'tensorflow/islr-models'

# 填入你那 4 个模型完整的文件名
WEIGHT_PATHS = [
    os.path.join(MODEL_DIR, 'islr-fp16-192-8-seed42-foldall-last.h5'),
    os.path.join(MODEL_DIR, 'islr-fp16-192-8-seed43-foldall-last.h5'),
    os.path.join(MODEL_DIR, 'islr-fp16-192-8-seed44-foldall-last.h5'),
    os.path.join(MODEL_DIR, 'islr-fp16-192-8-seed45-foldall-last.h5'),
    
]

# 检查一下文件是否真的在那里（调试用）
for p in WEIGHT_PATHS:
    if os.path.exists(p):
        print(f"✅ 找到模型: {p}")
    else:
        print(f"❌ 找不到模型: {p}，请检查文件名是否拼写正确")

✅ 找到模型: tensorflow/islr-models/islr-fp16-192-8-seed42-foldall-last.h5
✅ 找到模型: tensorflow/islr-models/islr-fp16-192-8-seed43-foldall-last.h5
✅ 找到模型: tensorflow/islr-models/islr-fp16-192-8-seed44-foldall-last.h5
✅ 找到模型: tensorflow/islr-models/islr-fp16-192-8-seed45-foldall-last.h5


In [11]:
SIGN_MAP_PATH = 'sign_to_prediction_index_map.json'
TEST_DATA_DIR = 'test_data'

In [12]:
# --- 初始化集成模型 ---
print("正在加载集成模型（4个模型聚合）...")
ensemble = ASLEnsemble(WEIGHT_PATHS, SIGN_MAP_PATH)

results = []

正在加载集成模型（4个模型聚合）...


/opt/anaconda3/envs/asl_env/lib/python3.9/site-packages/keras/src/saving/saving_api.py:282: UserWarning: Skipping loading of weights for layer #66 (named late_dropout) due to mismatch in number of weights. Layer expects 0 weight(s). Received 1 saved weight(s)
  legacy_h5_format.load_weights_from_hdf5_group_by_name(


✅ 模型 islr-fp16-192-8-seed42-foldall-last.h5 权重加载逻辑处理完成
✅ 模型 islr-fp16-192-8-seed43-foldall-last.h5 权重加载逻辑处理完成
✅ 模型 islr-fp16-192-8-seed44-foldall-last.h5 权重加载逻辑处理完成
✅ 模型 islr-fp16-192-8-seed45-foldall-last.h5 权重加载逻辑处理完成


In [13]:
# 获取 test_data 下的所有子文件夹（即每个单词名）
test_folders = [d for d in os.listdir(TEST_DATA_DIR) if os.path.isdir(os.path.join(TEST_DATA_DIR, d))]

print(f"🚀 开始评估，检测到 {len(test_folders)} 个单词文件夹...")


🚀 开始评估，检测到 22 个单词文件夹...


In [14]:
print("🚀 启动强制适配模式评估...")

for folder in test_folders:
    folder_path = os.path.join(TEST_DATA_DIR, folder)
    video_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.mp4', '.avi', '.mov'))]
    
    for v_file in video_files:
        v_path = os.path.join(folder_path, v_file)
        try:
            # 1. 提取骨架 (T, 543, 3)
            raw_landmarks = extract_landmarks_from_video(v_path)
            frame_count = len(raw_landmarks)
            
            # 2. 这里的预处理如果不稳，我们直接手动处理形状
            # 将 (T, 543, 3) 转换为模型需要的 (1, MAX_LEN, CHANNELS)
            input_tensor = prepare_single_sample(raw_landmarks)
            
            # 3. 预测
            pred_data = ensemble.predict(raw_landmarks, topk=5)
            
            actual = folder.lower()
            predicted = pred_data['predicted_label'].lower()
            
            results.append({
                'Actual': folder,
                'Predicted': pred_data['predicted_label'],
                'Is_Correct': 1 if actual == predicted else 0,
                'File': v_file,
                'Frames': frame_count,
                'Status': "✅" if actual == predicted else "❌"
            })
            print(f"[{results[-1]['Status']}] {folder} -> {pred_data['predicted_label']} ({frame_count}帧)")

        except Exception as e:
            # 如果还是报错，打印出到底是哪一行
            import traceback
            print(f"⚠️ 视频 {v_file} 失败原因:")
            traceback.print_exc() 
            results.append({
                'Actual': folder, 'Predicted': 'ERROR', 'Is_Correct': 0,
                'File': v_file, 'Frames': 0, 'Status': "⚠️"
            })

# 打印最终结果
df_final = pd.DataFrame(results)
if not df_final[df_final['Predicted'] != 'ERROR'].empty:
    acc = df_final[df_final['Predicted'] != 'ERROR']['Is_Correct'].mean()
    print(f"\n📊 最终成功率: {acc*100:.2f}%")

🚀 启动强制适配模式评估...


I0000 00:00:1778255595.202421 135339897 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 11, prefix = pthread-default
I0000 00:00:1778255595.282137 135339897 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778255595.323414 135339899 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255595.332706 135339901 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255595.333507 135339904 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255595.333512 135339908 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature infe

[❌] please -> after (86帧)


I0000 00:00:1778255596.829120 135339971 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1778255596.862923 135339977 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255596.867656 135339974 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255596.868407 135339975 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255596.868410 135339982 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255596.869190 135339977 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabl

[❌] friend -> after (72帧)


...

[❌] drink -> after (39帧)


I0000 00:00:1778255619.210512 135340800 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1778255619.244885 135340805 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255619.250146 135340807 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255619.250829 135340803 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255619.250873 135340807 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255619.251853 135340811 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabl

[❌] excuse_me -> after (67帧)


I0000 00:00:1778255620.376000 135340849 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1778255620.411066 135340853 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255620.415581 135340858 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255620.416314 135340860 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255620.416325 135340854 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778255620.417255 135340853 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabl

[❌] help -> after (75帧)

📊 最终成功率: 0.00%


In [ ]:
# --- 统计报表 ---
if results:
    df = pd.DataFrame(results)
    top1_acc = df['Is_Correct'].mean()
    top3_acc = df['In_Top3'].mean()
    
    print("\n" + "="*30)
    print("✨ 评估任务完成！")
    print(f"📊 总体 Top-1 准确率: {top1_acc*100:.2f}%")
    print(f"📊 总体 Top-3 准确率: {top3_acc*100:.2f}%")
    print("="*30)
    
    # 保存详细报告
    df.to_csv('asl_evaluation_report.csv', index=False, encoding='utf-8-sig')
    print("详细结果已保存至: asl_evaluation_report.csv")
else:
    print("❌ 未检测到有效视频，请检查 test_data 路径。")